# 04. 무패턴 진단 — ① 돈의 출처(상류) · ② 같은 분 같은 쌍

학습 없이 SQL 집계만 한다. 새 피처를 만들기 전에 **신호가 있는지, 있으면 얼마나 순도 높은 구간을 만드는지** 먼저 본다.

| 진단 | 질문 | 계산 |
|---|---|---|
| ① 상류 출처 | 송신 계좌가 이 거래 **전에** 세탁 거래로 돈을 받았나? 얼마나 최근에? | 송신 계좌의 마지막 패턴 양성 입금 / 무패턴 양성 입금 시각 → 경과 시간 구간 |
| ② 같은 분 같은 쌍 | 처음(또는 거의 처음) 보는 쌍이 같은 분에 또 보내면 얼마나 세탁인가? | 쌍의 과거 건수(같은 분 제외) × 같은 분 같은 쌍 다른 거래 수 |

- 라벨: `cls` 0 정상 · 1 패턴 양성(`patterns.parquet` 연결) · 2 무패턴 양성
- 이력: 2022-08-01 ~ 10-30 전체 거래(패턴 포함). 집계 대상: 우리 분할 Train 8/8~10/2 · Val 10/3~10/16 · Test 10/17~10/30
- **같은 분은 '과거'로 안 센다** (분 단위 데이터라 같은 분 안의 순서를 모름). ①은 `입금 시각 < 거래 시각`
- ①은 진단용으로 **라벨을 직접 쓴다**. 피처로 쓸 때는 라벨 대신 1단계 모델 점수(전날까지)로 바꿔야 한다
- 집중 송신 집단(`hot`): 팀원 문서의 "은행 70 송신 계좌 15개" — 무패턴을 많이 보낸 송신 계좌. 이 집단은 거래·계좌 피처로 정상과 구분이 안 된다고 보고됐으므로 **따로 나눠 본다**

## 0. 설정

GPU 서버 Jupyter, `code/nonpaten_vs_normal` 에서 위에서부터 실행. 필요한 입력은 `code/01_prepare_parquet.ipynb` 출력 세 개다.

```text
parquets/HI-Large/transactions.parquet
parquets/HI-Large/patterns.parquet
parquets/HI-Large/kyc.parquet
```

중간 테이블은 `parquets/HI-Large/nonpaten_vs_normal/04_diagnosis/work.duckdb` 에 쌓는다. 커널이 죽어도 다시 돌리면 **이미 만든 단계는 건너뛴다** (`REBUILD=True` 면 새로 만듦).

In [1]:
# 현재 Jupyter 커널에 패키지가 없을 때만 주석을 해제해 실행합니다.
# !pip install --break-system-packages --no-cache-dir -q duckdb pandas

In [2]:
from pathlib import Path
import json, time
import duckdb, numpy as np, pandas as pd
from IPython.display import display

DATASET = "HI-Large"
PROJECT_ROOT = None            # 자동 탐색 실패 시 Path("/workspace/money_laundry_ai_")
THREADS, MEMORY = 4, "8GB"     # DuckDB (넘치면 work 폴더 tmp/ 로 내려 씀)
REBUILD = False                # True: 중간 테이블 새로 만들기

HISTORY = ("2022-08-01", "2022-10-31")   # 이력으로 쓰는 거래 [시작, 끝) — 팀원 분석 기간과 같음
SPLITS = {"train": ("2022-08-08", "2022-10-02"),     # 끝 날짜 포함
          "val":   ("2022-10-03", "2022-10-16"),
          "test":  ("2022-10-17", "2022-10-30")}
HOT_MIN = 300                  # 이력 기간에 무패턴을 이 건수 이상 보낸 송신 계좌 = 집중 송신 집단
TEAM = {"nonpattern": 82_260, "hot_accounts": 15, "hot_nonpattern": 22_979}   # 팀원 문서 수치 (비교용)

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)


def find_root():
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "parquets" / DATASET / "transactions.parquet").is_file():
            return c
    raise FileNotFoundError("transactions.parquet 를 찾지 못했습니다. PROJECT_ROOT 를 지정하세요.")


ROOT = Path(PROJECT_ROOT).expanduser().resolve() if PROJECT_ROOT else find_root()
BASE = ROOT / "parquets" / DATASET
TX, PAT, KYC = BASE / "transactions.parquet", BASE / "patterns.parquet", BASE / "kyc.parquet"
OUT = BASE / "nonpaten_vs_normal" / "04_diagnosis"
for p in (TX, PAT, KYC):
    assert p.is_file(), p
OUT.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("OUT :", OUT)

ROOT: /workspace/money_laundry_ai_
OUT : /workspace/money_laundry_ai_/parquets/HI-Large/nonpaten_vs_normal/04_diagnosis


In [3]:
# DuckDB 연결 (파일 DB — 큰 중간 테이블을 디스크에 둔다)
con = duckdb.connect(str(OUT / "work.duckdb"))
con.execute(f"SET threads={THREADS}")
con.execute(f"SET memory_limit='{MEMORY}'")
con.execute(f"SET temp_directory='{OUT / 'tmp'}'")
con.execute("SET preserve_insertion_order=false")


def q(v):
    return "'" + str(v).replace("'", "''") + "'"


def has(name):
    return con.execute("SELECT count(*) FROM duckdb_tables() WHERE table_name = ?", [name]).fetchone()[0] > 0


def step(name, sql):
    """CREATE TABLE name AS sql — 이미 있으면 건너뜀"""
    if has(name) and not REBUILD:
        print(f"{name:12s} 이미 있음 — 건너뜀 ({con.execute(f'SELECT count(*) FROM {name}').fetchone()[0]:,} 행)")
        return
    t = time.perf_counter()
    con.execute(f"CREATE OR REPLACE TABLE {name} AS {sql}")
    n = con.execute(f"SELECT count(*) FROM {name}").fetchone()[0]
    print(f"{name:12s} {n:>13,} 행  ({time.perf_counter() - t:,.1f}s)", flush=True)


def split_sql(col="ts"):
    """분할 이름 (분할 밖이면 NULL)"""
    return "CASE " + " ".join(
        f"WHEN {col} >= TIMESTAMP {q(a)} AND {col} < DATE {q(b)} + INTERVAL 1 DAY THEN {q(s)}" for s, (a, b) in SPLITS.items()
    ) + " END"


print(split_sql())

CASE WHEN ts >= TIMESTAMP '2022-08-08' AND ts < DATE '2022-10-02' + INTERVAL 1 DAY THEN 'train' WHEN ts >= TIMESTAMP '2022-10-03' AND ts < DATE '2022-10-16' + INTERVAL 1 DAY THEN 'val' WHEN ts >= TIMESTAMP '2022-10-17' AND ts < DATE '2022-10-30' + INTERVAL 1 DAY THEN 'test' END


## 1. 거래 + 라벨 `tx`

필요한 열만 남긴다: `ts, src, dst, cls`. 패턴 연결은 01 노트북과 같이 `edge_id` 로 한다.

In [4]:
step("tx", f"""
    SELECT t.ts, t.src, t.dst,
           (CASE WHEN NOT t.is_laundering THEN 0 WHEN p.edge_id IS NOT NULL THEN 1 ELSE 2 END)::TINYINT AS cls
    FROM read_parquet({q(TX)}) t
    LEFT JOIN (SELECT DISTINCT edge_id FROM read_parquet({q(PAT)}) WHERE NOT unmatched) p ON p.edge_id = t.edge_id
    WHERE t.ts >= TIMESTAMP {q(HISTORY[0])} AND t.ts < TIMESTAMP {q(HISTORY[1])}
""")
# 패턴에 연결됐는데 정상 라벨인 거래가 없어야 한다
bad = con.execute(f"""SELECT count(*) FROM read_parquet({q(TX)}) t JOIN read_parquet({q(PAT)}) p ON p.edge_id = t.edge_id
                      WHERE NOT p.unmatched AND NOT t.is_laundering""").fetchone()[0]
assert bad == 0, f"패턴 연결 정상 거래 {bad:,}"

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

tx             168,490,912 행  (10.1s)


In [5]:
# 구간 × 라벨 건수 (이력만 = 8/1~8/7, 10/31 은 집계 대상 아님)
counts = con.execute(f"""
    SELECT coalesce({split_sql()}, '(이력만)') AS split,
           count(*) FILTER (WHERE cls = 0) AS normal, count(*) FILTER (WHERE cls = 1) AS pattern,
           count(*) FILTER (WHERE cls = 2) AS nonpattern, count(*) AS rows
    FROM tx GROUP BY 1 ORDER BY min(ts)
""").df().set_index("split")
counts.loc["합계"] = counts.sum()
counts["nonpattern_rate_%"] = (100 * counts.nonpattern / counts.rows).round(4)
display(counts)
print(f"이력 기간 무패턴 {int(counts.loc['합계','nonpattern']):,}  (팀원 문서 {TEAM['nonpattern']:,})")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,normal,pattern,nonpattern,rows,nonpattern_rate_%
split,,,,,
(이력만),14925801,1795,5411,14933007,0.0362
train,102225958,62889,47782,102336629,0.0467
val,24379251,20100,13695,24413046,0.0561
test,26773770,20187,14273,26808230,0.0532
합계,168304780,104971,81161,168490912,0.0482


이력 기간 무패턴 81,161  (팀원 문서 82,260)


## 2. 집중 송신 집단 `hot`

송신 계좌별 무패턴 건수. 팀원 문서: 은행 70 의 송신 계좌 15개가 무패턴 22,979건(약 28%)을 보냈고, 같은 계좌들이 정상 거래를 1,620만 건 보냈다.
분포를 보고 `HOT_MIN` 이 그 집단을 정확히 가르는지 확인한다.

In [6]:
step("sender_np", f"""
    SELECT t.src, k.bank_id, k.bank_name, k.entity_id, k.entity_name,
           count(*) FILTER (WHERE cls = 2) AS nonpattern, count(*) FILTER (WHERE cls = 1) AS pattern,
           count(*) FILTER (WHERE cls = 0) AS normal
    FROM tx t JOIN read_parquet({q(KYC)}) k ON k.node_id = t.src
    GROUP BY ALL HAVING count(*) FILTER (WHERE cls = 2) > 0
""")
# 송신 계좌 수 분포 (무패턴 건수 구간별)
display(con.execute("""
    SELECT CASE WHEN nonpattern = 1 THEN '1' WHEN nonpattern < 10 THEN '2~9' WHEN nonpattern < 100 THEN '10~99'
                WHEN nonpattern < 1000 THEN '100~999' ELSE '1000+' END AS nonpattern_per_sender,
           count(*) AS senders, sum(nonpattern) AS nonpattern, sum(normal) AS normal
    FROM sender_np GROUP BY 1 ORDER BY min(nonpattern)
""").df())
con.execute("SELECT * FROM sender_np ORDER BY nonpattern DESC LIMIT 30").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

sender_np           49,767 행  (34.4s)


,nonpattern_per_sender,senders,nonpattern,normal
0,1,42798,42798.0,2735664.0
1,2~9,6954,15677.0,463250.0
2,100~999,11,6571.0,4666183.0
3,1000+,4,16115.0,11334325.0


,src,bank_id,bank_name,entity_id,entity_name,nonpattern,pattern,normal
0,47308,70,National Bank of Indianapolis,2AA06BDDD80,Sole Proprietorship #136,8215,0,5819382
1,47309,70,National Bank of Indianapolis,2AA0482E530,Sole Proprietorship #51981,5275,0,3669838
2,47310,70,National Bank of Indianapolis,2AA053ACE80,Corporation #95138,1559,0,1127138
3,47320,70,National Bank of Indianapolis,2AA06957E70,Corporation #135412,1066,0,717967
4,47315,70,National Bank of Indianapolis,2AA05ECA9F0,Partnership #118218,827,0,552996
5,47316,70,National Bank of Indianapolis,2AA06B59A80,Partnership #22408,715,0,474727
6,47314,70,National Bank of Indianapolis,2AA05E8AF30,Corporation #115137,710,0,518656
7,47313,70,National Bank of Indianapolis,2AA05B71010,Sole Proprietorship #110711,695,0,491342
8,47311,70,National Bank of Indianapolis,2AA057E88F0,Partnership #18556,643,0,435888
9,47319,70,National Bank of Indianapolis,2AA06293150,Corporation #132542,631,0,429611


In [7]:
# HOT_MIN 을 바꾸면 바로 반영되도록 매번 새로 만든다
con.execute(f"CREATE OR REPLACE TABLE hot AS SELECT src FROM sender_np WHERE nonpattern >= {HOT_MIN}")
h = con.execute("""SELECT count(*) AS accounts, sum(nonpattern) AS nonpattern, sum(normal) AS normal, sum(pattern) AS pattern,
                          count(DISTINCT bank_id) AS banks, list(DISTINCT bank_id) AS bank_ids, count(DISTINCT entity_id) AS entities
                   FROM sender_np JOIN hot USING (src)""").df()
display(h)
print(f"팀원 문서: 계좌 {TEAM['hot_accounts']}개, 무패턴 {TEAM['hot_nonpattern']:,}건 — 다르면 HOT_MIN 을 조정")

,accounts,nonpattern,normal,pattern,banks,bank_ids,entities
0,15,22686.0,16000508.0,0.0,1,[70],15


팀원 문서: 계좌 15개, 무패턴 22,979건 — 다르면 HOT_MIN 을 조정


## 3. ① 상류 출처 — 송신 계좌가 마지막으로 세탁 입금을 받은 뒤 얼마나 지났나

`(src, ts)` 마다 `src` 가 **받은** 패턴 양성 거래 / 무패턴 양성 거래 중 `ts` 보다 이전 분의 마지막 시각을 ASOF JOIN 으로 붙인다.

경과 구간: `≤1시간` · `≤1일` · `≤7일` · `≤30일` · `>30일` · `없음`(이력 시작 8/1 이후 받은 적 없음)

- **패턴 입금**: 피처로 바꿀 수 있는 쪽 (1단계 모델이 패턴은 거의 다 잡으므로 점수로 대체 가능)
- **무패턴 입금**: 라벨 생성 방식 확인용 (무패턴이 무패턴을 낳는가). 피처로는 못 씀

In [8]:
# 세탁 입금 시각 (받은 계좌 기준, 분 단위 중복 제거)
step("in_pat", "SELECT DISTINCT dst AS acct, ts FROM tx WHERE cls = 1")
step("in_np",  "SELECT DISTINCT dst AS acct, ts FROM tx WHERE cls = 2")
# 집계 대상 분할의 (송신 계좌, 분)
step("src_ts", f"SELECT DISTINCT src, ts FROM tx WHERE ({split_sql()}) IS NOT NULL")
# 직전(이전 분) 세탁 입금 시각
step("up", """
    SELECT s.src, s.ts, p.ts AS last_pat_in, r.ts AS last_np_in
    FROM src_ts s
    ASOF LEFT JOIN in_pat p ON s.src = p.acct AND s.ts > p.ts
    ASOF LEFT JOIN in_np  r ON s.src = r.acct AND s.ts > r.ts
""")
con.execute("SELECT * FROM up WHERE last_pat_in IS NOT NULL LIMIT 5").df()

in_pat             104,966 행  (0.2s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

in_np               81,127 행  (2.2s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

src_ts         134,991,763 행  (9.9s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

up             134,991,763 행  (33.0s)


,src,ts,last_pat_in,last_np_in
0,685015,2022-10-08 21:10:00,2022-10-08 15:53:00,2022-08-25 06:42:00
1,685015,2022-10-08 21:22:00,2022-10-08 15:53:00,2022-08-25 06:42:00
2,685015,2022-10-09 19:16:00,2022-10-08 15:53:00,2022-08-25 06:42:00
3,685015,2022-10-09 19:26:00,2022-10-08 15:53:00,2022-08-25 06:42:00
4,685015,2022-10-10 21:53:00,2022-10-08 15:53:00,2022-08-25 06:42:00


## 4. ② 같은 분 같은 쌍 — 쌍 과거 건수 × 같은 분 다른 거래 수

`(src, dst, 분)` 칸으로 묶은 뒤 쌍별 누적합으로 **같은 분 이전까지의 쌍 건수**(`prior_n`)를 구한다.
칸 안의 거래 수가 `n` 이면 각 거래의 같은 분 같은 쌍 다른 거래 수는 `n − 1` 이다. 이력 8/1 부터, 패턴 거래 포함.

In [9]:
step("pair_cells", """
    SELECT src, dst, ts, count(*)::INT AS n,
           count(*) FILTER (WHERE cls = 0)::INT AS n0, count(*) FILTER (WHERE cls = 1)::INT AS n1,
           count(*) FILTER (WHERE cls = 2)::INT AS n2
    FROM tx GROUP BY ALL
""")
step("pair_cum", """
    SELECT *, coalesce(sum(n) OVER (PARTITION BY src, dst ORDER BY ts
                                    ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING), 0)::BIGINT AS prior_n
    FROM pair_cells
""")
con.execute("SELECT * FROM pair_cum WHERE n > 1 ORDER BY src, dst, ts LIMIT 8").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pair_cells     164,929,785 행  (24.6s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pair_cum       164,929,785 행  (47.0s)


,src,dst,ts,n,n0,n1,n2,prior_n
0,0,0,2022-08-05 04:00:00,2,2,0,0,12
1,0,0,2022-09-07 03:45:00,2,2,0,0,68
2,0,0,2022-10-12 17:56:00,2,2,0,0,131
3,0,65675,2022-08-09 01:28:00,2,2,0,0,12
4,0,65675,2022-09-07 21:02:00,2,2,0,0,54
5,0,65675,2022-09-20 05:13:00,2,2,0,0,72
6,0,86029,2022-09-05 06:49:00,2,2,0,0,70
7,0,86029,2022-09-13 02:15:00,2,2,0,0,86


## 5. 한 표로 모으기 `agg`

분할 × hot × ①패턴 입금 구간 × ①무패턴 입금 구간 × ②과거 건수 구간 × ②같은 분 구간 → 라벨별 건수.
이후 표는 모두 이 작은 표에서 pandas 로 만든다.

In [10]:
GAP = ["≤1시간", "≤1일", "≤7일", "≤30일", ">30일", "없음"]
PRIOR = ["0", "1", "2~5", "6~20", "21+"]
SAME = ["0", "1", "2~4", "5+"]


def gap_code(col):
    m = f"date_diff('minute', {col}, c.ts)"
    return (f"CASE WHEN {col} IS NULL THEN 5 WHEN {m} <= 60 THEN 0 WHEN {m} <= 1440 THEN 1 "
            f"WHEN {m} <= 10080 THEN 2 WHEN {m} <= 43200 THEN 3 ELSE 4 END")


agg = con.execute(f"""
    SELECT {split_sql('c.ts')} AS split, (h.src IS NOT NULL) AS hot,
           {gap_code('u.last_pat_in')} AS g_pat, {gap_code('u.last_np_in')} AS g_np,
           CASE WHEN prior_n = 0 THEN 0 WHEN prior_n = 1 THEN 1 WHEN prior_n <= 5 THEN 2 WHEN prior_n <= 20 THEN 3 ELSE 4 END AS prior_b,
           CASE WHEN n = 1 THEN 0 WHEN n = 2 THEN 1 WHEN n <= 5 THEN 2 ELSE 3 END AS same_b,
           sum(n0) AS n0, sum(n1) AS n1, sum(n2) AS n2
    FROM pair_cum c
    JOIN up u ON u.src = c.src AND u.ts = c.ts
    LEFT JOIN hot h ON h.src = c.src
    GROUP BY ALL
""").df()
agg["split"] = pd.Categorical(agg.split, list(SPLITS))
for col, labels in [("g_pat", GAP), ("g_np", GAP), ("prior_b", PRIOR), ("same_b", SAME)]:
    agg[col] = pd.Categorical.from_codes(agg[col].astype(int), labels)
agg.to_parquet(OUT / "agg.parquet", index=False)

# 분할별 건수가 1번 표와 같아야 한다 (빠진 거래 없음)
chk = agg.groupby("split", observed=True)[["n0", "n1", "n2"]].sum()
assert (chk.values == counts.loc[list(SPLITS), ["normal", "pattern", "nonpattern"]].values).all(), chk
print(len(agg), "칸 · 분할별 건수 1번 표와 일치")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

954 칸 · 분할별 건수 1번 표와 일치


In [11]:
def table(df, by):
    """구간별 건수와 순도.
    무패턴_비율 = 무패턴 / (정상 + 무패턴)  (패턴 제외, 03 과 같은 기준)
    양성_비율   = (패턴 + 무패턴) / 전체     (이진 모델 운영점 기준)
    무패턴_비중 / 정상_비중 = 이 구간에 들어간 무패턴·정상의 비율, 배수 = 둘의 비"""
    t = df.groupby(by, observed=True)[["n0", "n1", "n2"]].sum()
    t["rows"] = t.n0 + t.n1 + t.n2
    t["무패턴_비율_%"] = 100 * t.n2 / (t.n0 + t.n2).clip(lower=1)
    t["양성_비율_%"] = 100 * (t.n1 + t.n2) / t.rows.clip(lower=1)
    t["무패턴_비중_%"] = 100 * t.n2 / max(t.n2.sum(), 1)
    t["정상_비중_%"] = 100 * t.n0 / max(t.n0.sum(), 1)
    t["배수"] = t["무패턴_비중_%"] / t["정상_비중_%"].replace(0, np.nan)
    return t.round(3)


def rule_curve(df, by, top=12):
    """구간을 양성 비율 높은 순으로 하나씩 더해 갈 때 P / 전체 R / 무패턴 R — 이 피처 하나만으로 만든 규칙의 상한"""
    t = df.groupby(by, observed=True)[["n0", "n1", "n2"]].sum()
    t = t[(t.n0 + t.n1 + t.n2) > 0]
    t["rows"] = t.n0 + t.n1 + t.n2
    t["pos"] = t.n1 + t.n2
    t = t.assign(rate=t.pos / t.rows).sort_values("rate", ascending=False).drop(columns="rate")
    cum = t.cumsum()
    t["P_%"] = 100 * cum.pos / cum.rows
    t["R_전체_%"] = 100 * cum.pos / t.pos.sum()
    t["R_무패턴_%"] = 100 * cum.n2 / max(t.n2.sum(), 1)
    return t.head(top).round(3)

## 6. ① 결과 — 상류 출처

먼저 전체 분할 합계를 hot / 나머지로 나눠 본다. 읽는 법:
- `무패턴_비중` 과 `정상_비중` 이 크게 다르면(배수 ≫ 1) 신호가 있다
- 그 구간의 `무패턴_비율` 이 기준선(1번 표 약 0.05%)보다 몇 배인지가 순도

In [12]:
# 패턴 양성 입금 기준 (피처로 바꿀 수 있는 쪽)
for hot, name in [(True, "hot 집단"), (False, "나머지")]:
    print(f"── {name} · 송신 계좌의 직전 패턴 양성 입금")
    display(table(agg[agg.hot == hot], "g_pat"))

── hot 집단 · 송신 계좌의 직전 패턴 양성 입금


,n0,n1,n2,rows,무패턴_비율_%,양성_비율_%,무패턴_비중_%,정상_비중_%,배수
g_pat,,,,,,,,,
없음,14720427.0,0.0,20857.0,14741284.0,0.141,0.141,100.0,100.0,1.0


── 나머지 · 송신 계좌의 직전 패턴 양성 입금


,n0,n1,n2,rows,무패턴_비율_%,양성_비율_%,무패턴_비중_%,정상_비중_%,배수
g_pat,,,,,,,,,
≤1시간,10086.0,785.0,6.0,10877.0,0.059,7.272,0.011,0.007,1.503
≤1일,208628.0,11073.0,135.0,219836.0,0.065,5.098,0.246,0.150,1.635
≤7일,940539.0,17948.0,703.0,959190.0,0.075,1.944,1.281,0.678,1.888
≤30일,2273105.0,19957.0,1709.0,2294771.0,0.075,0.944,3.113,1.639,1.899
>30일,1630609.0,1790.0,1293.0,1633692.0,0.079,0.189,2.355,1.176,2.003
없음,133595585.0,51623.0,51047.0,133698255.0,0.038,0.077,92.994,96.349,0.965


In [13]:
# 무패턴 양성 입금 기준 (라벨 생성 확인용)
for hot, name in [(True, "hot 집단"), (False, "나머지")]:
    print(f"── {name} · 송신 계좌의 직전 무패턴 양성 입금")
    display(table(agg[agg.hot == hot], "g_np"))

── hot 집단 · 송신 계좌의 직전 무패턴 양성 입금


,n0,n1,n2,rows,무패턴_비율_%,양성_비율_%,무패턴_비중_%,정상_비중_%,배수
g_np,,,,,,,,,
≤1시간,38247.0,0.0,54.0,38301.0,0.141,0.141,0.259,0.260,0.996
≤1일,540953.0,0.0,772.0,541725.0,0.143,0.143,3.701,3.675,1.007
≤7일,3074162.0,0.0,4297.0,3078459.0,0.140,0.140,20.602,20.884,0.987
≤30일,4880736.0,0.0,7032.0,4887768.0,0.144,0.144,33.715,33.156,1.017
>30일,456310.0,0.0,703.0,457013.0,0.154,0.154,3.371,3.100,1.087
없음,5730019.0,0.0,7999.0,5738018.0,0.139,0.139,38.352,38.926,0.985


── 나머지 · 송신 계좌의 직전 무패턴 양성 입금


,n0,n1,n2,rows,무패턴_비율_%,양성_비율_%,무패턴_비중_%,정상_비중_%,배수
g_np,,,,,,,,,
≤1시간,3127.0,4.0,2.0,3133.0,0.064,0.192,0.004,0.002,1.616
≤1일,69807.0,51.0,147.0,70005.0,0.210,0.283,0.268,0.050,5.319
≤7일,407965.0,473.0,930.0,409368.0,0.227,0.343,1.694,0.294,5.758
≤30일,1291418.0,1490.0,2608.0,1295516.0,0.202,0.316,4.751,0.931,5.101
>30일,1285651.0,1394.0,2424.0,1289469.0,0.188,0.296,4.416,0.927,4.763
없음,135600584.0,99764.0,48782.0,135749130.0,0.036,0.109,88.867,97.795,0.909


In [14]:
# hot 계좌들은 세탁 입금을 받는가 — 계좌별 받은 패턴·무패턴 양성 건수
con.execute("""
    WITH recv AS (SELECT dst AS src, count(*) FILTER (WHERE cls = 1) AS recv_pattern,
                         count(*) FILTER (WHERE cls = 2) AS recv_nonpattern, count(*) AS recv_all
                  FROM tx WHERE dst IN (SELECT src FROM hot) GROUP BY dst)
    SELECT s.src, s.bank_id, s.nonpattern AS sent_nonpattern, s.normal AS sent_normal,
           coalesce(r.recv_pattern, 0) AS recv_pattern, coalesce(r.recv_nonpattern, 0) AS recv_nonpattern,
           coalesce(r.recv_all, 0) AS recv_all
    FROM sender_np s JOIN hot USING (src) LEFT JOIN recv r USING (src) ORDER BY sent_nonpattern DESC
""").df()

,src,bank_id,sent_nonpattern,sent_normal,recv_pattern,recv_nonpattern,recv_all
0,47308,70,8215,5819382,0,7,29637
1,47309,70,5275,3669838,0,3,18644
2,47310,70,1559,1127138,0,1,6660
3,47320,70,1066,717967,0,2,3700
4,47315,70,827,552996,0,1,2758
5,47316,70,715,474727,0,0,2533
6,47314,70,710,518656,0,2,2407
7,47313,70,695,491342,0,0,2377
8,47311,70,643,435888,0,0,2126
9,47319,70,631,429611,0,0,2142


In [15]:
# 분할별로 같은 경향인가 (나머지 집단, 패턴 입금 기준)
rest = agg[~agg.hot]
pd.concat({s: table(rest[rest.split == s], "g_pat")[["n2", "무패턴_비율_%", "무패턴_비중_%", "정상_비중_%", "배수"]]
           for s in SPLITS}, axis=1)

train                                      val                                     test                                 
            n2 무패턴_비율_% 무패턴_비중_% 정상_비중_%     배수      n2 무패턴_비율_% 무패턴_비중_% 정상_비중_%     배수      n2 무패턴_비율_% 무패턴_비중_% 정상_비중_%     배수
g_pat                                                                                                                            
≤1시간       2.0    0.034    0.006   0.006  0.913     1.0    0.049    0.010   0.009  1.054     3.0    0.143    0.028   0.009  3.263
≤1일       90.0    0.071    0.265   0.138  1.922    20.0    0.051    0.196   0.180  1.089    25.0    0.060    0.234   0.172  1.358
≤7일      399.0    0.071    1.174   0.605  1.942   152.0    0.082    1.488   0.840  1.771   152.0    0.077    1.421   0.813  1.748
≤30일     816.0    0.071    2.401   1.235  1.944   442.0    0.082    4.326   2.446  1.769   451.0    0.076    4.217   2.448  1.722
>30일     267.0    0.080    0.786   0.361  2.177   381.0    0.078    3.729   2.222  1.678   645.0    0.080    6.031   3.331  1.810
없음     32406.0    0.036   95.368  97.655  0.977  9222.0    0.045   90.252  94.304  0.957  9419.0    0.042   88.069  93.227  0.945

In [16]:
# 규칙 상한: 패턴 입금 구간 하나만으로 (분할 합계, 전체 집단)
rule_curve(agg, "g_pat")

,n0,n1,n2,rows,pos,P_%,R_전체_%,R_무패턴_%
g_pat,,,,,,,,
≤1시간,10086.0,785.0,6.0,10877.0,791.0,7.272,0.442,0.008
≤1일,208628.0,11073.0,135.0,219836.0,11208.0,5.201,6.706,0.186
≤7일,940539.0,17948.0,703.0,959190.0,18651.0,2.576,17.130,1.114
≤30일,2273105.0,19957.0,1709.0,2294771.0,21666.0,1.501,29.239,3.370
>30일,1630609.0,1790.0,1293.0,1633692.0,3083.0,1.082,30.962,5.077
없음,148316012.0,51623.0,71904.0,148439539.0,123527.0,0.117,100.000,100.000


## 7. ② 결과 — 같은 분 같은 쌍

03 결과(무패턴 데이터): 과거 0건 쌍이 같은 분에 또 보내면 무패턴 비율 17.9%, 과거 1건이면 49.1%.
여기서는 **패턴 거래도 포함한 전체 원장**이라 같은 분 다른 거래에 패턴 거래도 센다.

In [17]:
t = table(agg, ["prior_b", "same_b"])
print("무패턴 비율 % (패턴 제외) — 행: 쌍 과거 건수, 열: 같은 분 같은 쌍 다른 거래 수")
display(t["무패턴_비율_%"].unstack())
print("무패턴 건수")
display(t["n2"].unstack())
print("양성 비율 % (패턴 포함)")
display(t["양성_비율_%"].unstack())

무패턴 비율 % (패턴 제외) — 행: 쌍 과거 건수, 열: 같은 분 같은 쌍 다른 거래 수


same_b,0,1,2~4
prior_b,,,
0,0.547,17.653,NaN
1,0.452,49.057,NaN
2~5,0.131,0.547,0.000
6~20,0.039,0.027,0.000
21+,0.016,0.012,0.007


무패턴 건수


same_b,0,1,2~4
prior_b,,,
0,23308.0,2416.0,NaN
1,11827.0,1040.0,NaN
2~5,11033.0,761.0,0.0
6~20,6278.0,125.0,0.0
21+,18232.0,723.0,7.0


양성 비율 % (패턴 포함)


same_b,0,1,2~4
prior_b,,,
0,2.412,49.996,NaN
1,0.515,50.000,NaN
2~5,0.173,0.610,0.000
6~20,0.060,0.061,0.000
21+,0.019,0.015,0.007


In [18]:
# hot / 나머지
pd.concat({name: table(agg[agg.hot == hot], ["prior_b", "same_b"])[["n2", "무패턴_비율_%", "무패턴_비중_%"]]
           for hot, name in [(True, "hot"), (False, "나머지")]}, axis=1).query("same_b != '0'")

hot                       나머지                  
                   n2 무패턴_비율_% 무패턴_비중_%      n2 무패턴_비율_% 무패턴_비중_%
prior_b same_b                                                   
2~5     1        11.0    0.163    0.053   750.0    0.566    1.366
        2~4       0.0    0.000    0.000     0.0    0.000    0.000
6~20    1        50.0    0.111    0.240    75.0    0.018    0.137
        2~4       0.0    0.000    0.000     0.0    0.000    0.000
21+     1       719.0    0.137    3.447     4.0    0.000    0.007
        2~4       7.0    0.153    0.034     0.0    0.000    0.000
0       1         NaN      NaN      NaN  2416.0   17.653    4.401
1       1         NaN      NaN      NaN  1040.0   49.057    1.895

In [19]:
# 분할별 안정성 — 같은 분 1건 이상 칸만
pd.concat({s: table(agg[agg.split == s], ["prior_b", "same_b"])[["n2", "무패턴_비율_%", "양성_비율_%"]]
           for s in SPLITS}, axis=1).query("same_b != '0'")

train                     val                    test                 
                    n2 무패턴_비율_% 양성_비율_%     n2 무패턴_비율_% 양성_비율_%     n2 무패턴_비율_% 양성_비율_%
prior_b same_b                                                                         
0       1       1376.0   17.273  49.992  490.0   17.544  50.000  550.0   18.791  50.000
1       1        457.0   48.565  50.000  246.0   49.004  50.000  337.0   49.778  50.000
2~5     1        251.0    0.182   0.238  218.0   48.989  49.779  292.0   49.660  50.000
        2~4        0.0    0.000   0.000    NaN      NaN     NaN    NaN      NaN     NaN
6~20    1         69.0    0.018   0.052   19.0    0.071   0.109   37.0    0.099   0.123
        2~4        0.0    0.000   0.000    0.0    0.000   0.000    0.0    0.000   0.000
21+     1        480.0    0.013   0.015  115.0    0.011   0.015  128.0    0.012   0.015
        2~4        6.0    0.009   0.009    0.0    0.000   0.000    1.0    0.006   0.006

In [20]:
# 규칙 상한: (과거 건수, 같은 분) 칸만으로
rule_curve(agg, ["prior_b", "same_b"])

n0       n1       n2         rows       pos     P_%   R_전체_%  R_무패턴_%
prior_b same_b                                                                                
1       1            1080.0     40.0   1040.0       2160.0    1080.0  50.000    0.604    1.373
0       1           11270.0   8852.0   2416.0      22538.0   11268.0  49.996    6.901    4.562
        0         4234047.0  81363.0  23308.0    4338718.0  104671.0   2.682   65.401   35.332
2~5     1          138434.0     89.0    761.0     139284.0     850.0   2.618   65.876   36.337
1       0         2603520.0   1645.0  11827.0    2616992.0   13472.0   1.845   73.405   51.950
2~5     0         8399231.0   3554.0  11033.0    8413818.0   14587.0   0.939   81.558   66.515
6~20    1          455304.0    153.0    125.0     455582.0     278.0   0.914   81.713   66.680
        0        16123088.0   3464.0   6278.0   16132830.0    9742.0   0.485   87.158   74.968
21+     0       115521782.0   3886.0  18232.0  115543900.0   22118.0   0.121   99.519   99.036
        1         5784369.0    130.0    723.0    5785222.0     853.0   0.117   99.996   99.991
        2~4        100429.0      0.0      7.0     100436.0       7.0   0.117  100.000  100.000
2~5     2~4          1206.0      0.0      0.0       1206.0       0.0   0.117  100.000  100.000

## 8. ① × ② 겹침 — 두 신호가 같은 무패턴을 가리키나

무패턴 거래(분할 합계)를 "직전 7일 안에 패턴 양성 입금" × "같은 분 같은 쌍 1건 이상" 으로 나눈다. 많이 안 겹치면 둘 다 넣을 가치가 있다.

In [21]:
x = agg.assign(up7=agg.g_pat.isin(["≤1시간", "≤1일", "≤7일"]), same=agg.same_b != "0")
pd.concat({name: x[x.hot == hot].groupby(["up7", "same"])[["n0", "n2"]].sum().assign(
               무패턴_비율_pct=lambda d: (100 * d.n2 / (d.n0 + d.n2)).round(3),
               무패턴_비중_pct=lambda d: (100 * d.n2 / d.n2.sum()).round(2))
           for hot, name in [(True, "hot"), (False, "나머지")]}, axis=1)

hot                                         나머지                               
                     n0       n2 무패턴_비율_pct 무패턴_비중_pct           n0       n2 무패턴_비율_pct 무패턴_비중_pct
up7   same                                                                                        
False False  14138366.0  20070.0      0.142      96.23  131644355.0  49967.0      0.038      91.03
      True     582061.0    787.0      0.135       3.77    5854944.0   4082.0      0.070       7.44
True  False         NaN      NaN        NaN        NaN    1098947.0    641.0      0.058       1.17
      True          NaN      NaN        NaN        NaN      60306.0    203.0      0.335       0.37

## 9. 요약 저장 · 판단 기준

| 결과 | 다음 단계 |
|---|---|
| ① 나머지 집단에서 `≤1일`/`≤7일` 배수가 크고(예: 5배 이상) 무패턴 비중도 수 % 이상 | 05 에서 "직전 k일 고위험 입금" 피처로 만든다 (1단계 점수 사용) |
| ① hot 집단에서 무패턴만 세탁 입금 직후에 몰림 | hot 집단을 가를 유일한 단서 → 우선순위 1 |
| ① 배수 ≈ 1 | ① 접음 |
| ② 같은 분 칸 순도가 분할마다 비슷하게 유지 | 05 에 `pair_same_minute_other_n` 등 추가 |

In [22]:
summary = {}
for hot, name in [(True, "hot"), (False, "rest"), (None, "all")]:
    d = agg if hot is None else agg[agg.hot == hot]
    t = table(d, "g_pat").reindex(GAP).fillna(0)
    near = ["≤1시간", "≤1일", "≤7일"]
    summary[name] = {
        "nonpattern": int(d.n2.sum()), "normal": int(d.n0.sum()),
        "pat_in_7d_nonpattern_%": round(float(t.loc[near, "무패턴_비중_%"].sum()), 3),
        "pat_in_7d_normal_%": round(float(t.loc[near, "정상_비중_%"].sum()), 3),
        "same_minute_nonpattern_%": round(float(100 * d.loc[d.same_b != "0", "n2"].sum() / max(d.n2.sum(), 1)), 3),
        "same_minute_normal_%": round(float(100 * d.loc[d.same_b != "0", "n0"].sum() / max(d.n0.sum(), 1)), 3),
    }
summary = {"history": HISTORY, "splits": SPLITS, "hot_min": HOT_MIN, **summary}
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
table(agg, "g_pat").to_csv(OUT / "upstream_pattern_all.csv")
table(agg, ["prior_b", "same_b"]).to_csv(OUT / "same_minute_pair_all.csv")
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "history": [
    "2022-08-01",
    "2022-10-31"
  ],
  "splits": {
    "train": [
      "2022-08-08",
      "2022-10-02"
    ],
    "val": [
      "2022-10-03",
      "2022-10-16"
    ],
    "test": [
      "2022-10-17",
      "2022-10-30"
    ]
  },
  "hot_min": 300,
  "hot": {
    "nonpattern": 20857,
    "normal": 14720427,
    "pat_in_7d_nonpattern_%": 0.0,
    "pat_in_7d_normal_%": 0.0,
    "same_minute_nonpattern_%": 3.773,
    "same_minute_normal_%": 3.954
  },
  "rest": {
    "nonpattern": 54893,
    "normal": 138658552,
    "pat_in_7d_nonpattern_%": 1.538,
    "pat_in_7d_normal_%": 0.835,
    "same_minute_nonpattern_%": 7.806,
    "same_minute_normal_%": 4.266
  },
  "all": {
    "nonpattern": 75750,
    "normal": 153378979,
    "pat_in_7d_nonpattern_%": 1.114,
    "pat_in_7d_normal_%": 0.756,
    "same_minute_nonpattern_%": 6.696,
    "same_minute_normal_%": 4.236
  }
}


In [24]:
# 10. '양성 1 + 정상 1' 쌍둥이 칸 — 두 거래가 무엇이 다른가
con = duckdb.connect(str(OUT / "work.duckdb"))          # 마지막 셀에서 닫았으므로 다시 연다
con.execute(f"SET memory_limit='{MEMORY}'"); con.execute(f"SET temp_directory='{OUT / 'tmp'}'")
con.execute(f"""
    CREATE OR REPLACE TABLE twins AS
    WITH cells AS (SELECT src, dst, ts, prior_n FROM pair_cum WHERE n = 2 AND n0 = 1 AND n1 + n2 = 1),
    t AS (SELECT x.edge_id, x.ts, x.src, x.dst, x.paid, x.received, x.payment_currency, x.receiving_currency,
                 x.payment_format, x.is_laundering, c.prior_n
          FROM read_parquet({q(TX)}) x JOIN cells c ON c.src = x.src AND c.dst = x.dst AND c.ts = x.ts)
    SELECT p.src, p.dst, p.ts, p.prior_n, p.edge_id AS p_id, n.edge_id AS n_id,
           p.paid AS p_paid, n.paid AS n_paid, p.received AS p_recv, n.received AS n_recv,
           p.payment_currency AS p_pccy, n.payment_currency AS n_pccy,
           p.receiving_currency AS p_rccy, n.receiving_currency AS n_rccy,
           p.payment_format AS p_fmt, n.payment_format AS n_fmt
    FROM t p JOIN t n ON n.src = p.src AND n.dst = p.dst AND n.ts = p.ts AND p.is_laundering AND NOT n.is_laundering
""")
# 항목별 일치 비율 (과거 0~1건 / 2건 이상)
display(con.execute("""
    SELECT prior_n <= 1 AS new_pair, count(*) AS cells,
           avg((p_paid = n_paid)::INT) AS same_paid, avg((p_recv = n_recv)::INT) AS same_received,
           avg((p_pccy = n_pccy)::INT) AS same_pay_ccy, avg((p_rccy = n_rccy)::INT) AS same_recv_ccy,
           avg((p_fmt = n_fmt)::INT) AS same_format, avg((abs(p_id - n_id) = 1)::INT) AS adjacent_id
    FROM twins GROUP BY 1 ORDER BY 1 DESC
""").df())
# 결제방식·통화 조합
display(con.execute("""
    SELECT p_fmt, n_fmt, p_pccy = p_rccy AS p_same_ccy, n_pccy = n_rccy AS n_same_ccy, count(*) AS cells
    FROM twins GROUP BY ALL ORDER BY cells DESC LIMIT 15
""").df())
con.execute("SELECT * FROM twins ORDER BY ts LIMIT 10").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,new_pair,cells,same_paid,same_received,same_pay_ccy,same_recv_ccy,same_format,adjacent_id
0,True,12465,0.0,0.0,1.0,1.0,0.998396,1.0
1,False,2035,0.0,0.0,1.0,1.0,0.600000,1.0


,p_fmt,n_fmt,p_same_ccy,n_same_ccy,cells
0,ACH,ACH,True,True,13490
1,Credit Card,Cheque,True,True,224
2,Cheque,Credit Card,True,True,220
3,Cheque,Cash,True,True,144
4,Cheque,Cheque,True,True,121
5,Cash,Cheque,True,True,115
6,Credit Card,Cash,True,True,68
7,Cash,Credit Card,True,True,63
8,Bitcoin,Bitcoin,True,True,55


,src,dst,ts,prior_n,p_id,n_id,p_paid,n_paid,p_recv,n_recv,p_pccy,n_pccy,p_rccy,n_rccy,p_fmt,n_fmt
0,47313,1017357,2022-08-01 00:03:00,0,132567,132568,16201.55,1204434.22,16201.55,1204434.22,Ruble,Ruble,Ruble,Ruble,Credit Card,Cheque
1,47308,174878,2022-08-01 00:19:00,0,832731,832732,27815.14,128078.37,27815.14,128078.37,US Dollar,US Dollar,US Dollar,US Dollar,Cheque,Cash
2,47309,680277,2022-08-01 01:07:00,0,1396710,1396709,9894.00,352.78,9894.00,352.78,Euro,Euro,Euro,Euro,Cheque,Credit Card
3,47308,656958,2022-08-01 02:31:00,0,1584798,1584797,4641.93,1267.49,4641.93,1267.49,US Dollar,US Dollar,US Dollar,US Dollar,Cheque,Cash
4,47310,629949,2022-08-01 02:32:00,0,1587157,1587158,7356.73,14983.37,7356.73,14983.37,Yuan,Yuan,Yuan,Yuan,Credit Card,Cheque
5,1049700,1336490,2022-08-01 04:19:00,0,1826872,1826871,10146.62,3722.20,10146.62,3722.20,UK Pound,UK Pound,UK Pound,UK Pound,ACH,ACH
6,47309,1031094,2022-08-01 08:19:00,0,2360502,2360501,88606.79,24393.57,88606.79,24393.57,Euro,Euro,Euro,Euro,Cheque,Cash
7,47308,1176304,2022-08-01 08:35:00,0,2396619,2396618,3450.46,120.13,3450.46,120.13,US Dollar,US Dollar,US Dollar,US Dollar,Cheque,Cash
8,47310,863853,2022-08-01 10:01:00,0,2589106,2589105,206513.27,66365.54,206513.27,66365.54,Yuan,Yuan,Yuan,Yuan,Cash,Credit Card
9,153764,153764,2022-08-01 10:54:00,1,2707987,2707988,6302.59,9258.12,6302.59,9258.12,US Dollar,US Dollar,US Dollar,US Dollar,ACH,ACH


In [25]:
# 11. 쌍둥이 안에서 양성·정상을 가르는 단서 — 금액
con.execute(f"""
    CREATE OR REPLACE TABLE twin_amt AS
    WITH sent AS (SELECT src, payment_currency AS ccy, paid, min(ts) AS first_ts
                  FROM read_parquet({q(TX)}) WHERE src IN (SELECT src FROM twins) GROUP BY ALL),
         recv AS (SELECT dst, receiving_currency AS ccy, received, min(ts) AS first_ts
                  FROM read_parquet({q(TX)}) WHERE dst IN (SELECT dst FROM twins) GROUP BY ALL)
    SELECT w.*, (h.src IS NOT NULL) AS hot,
           coalesce(sp.first_ts < w.ts, false) AS p_sent_before,   -- 송신자가 이 통화·금액을 이전 분에 보낸 적 있나
           coalesce(sn.first_ts < w.ts, false) AS n_sent_before,
           coalesce(rp.first_ts < w.ts, false) AS p_recv_before,   -- 수신자가 이 통화·금액을 이전 분에 받은 적 있나
           coalesce(rn.first_ts < w.ts, false) AS n_recv_before
    FROM twins w
    LEFT JOIN hot h  ON h.src = w.src
    LEFT JOIN sent sp ON sp.src = w.src AND sp.ccy = w.p_pccy AND sp.paid = w.p_paid
    LEFT JOIN sent sn ON sn.src = w.src AND sn.ccy = w.n_pccy AND sn.paid = w.n_paid
    LEFT JOIN recv rp ON rp.dst = w.dst AND rp.ccy = w.p_rccy AND rp.received = w.p_recv
    LEFT JOIN recv rn ON rn.dst = w.dst AND rn.ccy = w.n_rccy AND rn.received = w.n_recv
""")
display(con.execute("""
    SELECT hot, (p_fmt = 'ACH' AND n_fmt = 'ACH') AS ach, prior_n <= 1 AS new_pair, count(*) AS cells,
           avg((p_paid > n_paid)::INT)            AS p_larger,
           avg(p_sent_before::INT) AS p_sent_before, avg(n_sent_before::INT) AS n_sent_before,
           avg(p_recv_before::INT) AS p_recv_before, avg(n_recv_before::INT) AS n_recv_before,
           avg((p_paid = round(p_paid))::INT)     AS p_whole, avg((n_paid = round(n_paid))::INT) AS n_whole,
           median(ln(p_paid / n_paid))            AS med_log_ratio
    FROM twin_amt GROUP BY ALL ORDER BY hot, ach DESC, new_pair DESC
""").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,hot,ach,new_pair,cells,p_larger,p_sent_before,n_sent_before,p_recv_before,n_recv_before,p_whole,n_whole,med_log_ratio
0,False,True,True,12327,0.700982,0.000000,0.00,0.001460,0.000081,0.010870,0.010303,1.962396
1,False,True,False,1163,0.366294,0.013758,0.00,0.016337,0.000000,0.008598,0.009458,-0.743301
2,False,False,True,118,0.161017,0.000000,0.00,0.008475,0.000000,0.008475,0.008475,-1.867552
3,False,False,False,50,0.220000,0.040000,0.04,0.040000,0.040000,0.020000,0.000000,-1.693609
4,True,False,True,20,0.500000,0.100000,0.00,0.000000,0.000000,0.050000,0.000000,0.501780
5,True,False,False,822,0.459854,1.000000,1.00,1.000000,1.000000,0.012165,0.010949,-0.119777


In [26]:
# 정리: 중간 테이블은 work.duckdb 에 남겨 둔다 (다시 실행 시 재사용). 지우려면 DELETE_WORK=True
DELETE_WORK = False
con.close()
if DELETE_WORK:
    (OUT / "work.duckdb").unlink(missing_ok=True)
    (OUT / "work.duckdb.wal").unlink(missing_ok=True)
print("완료. 결과:", OUT)

완료. 결과: /workspace/money_laundry_ai_/parquets/HI-Large/nonpaten_vs_normal/04_diagnosis
